<a href="https://colab.research.google.com/github/maricmello/Natural-Language-Processing/blob/main/Attention/04_Attention_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nome: Mariana Costa de Mello


## Instruções:

Treinar e medir a acurácia de um modelo de classificação binária usando o dataset do IMDB (20k/5k amostras de treino/validação).
O modelo deverá ter uma camada de auto-atenção simplificada.

Deverão ser entregues duas implementações da camada de auto-atenção.
1. Usando laços (ineficiente, mas bom para o aprendizado)
2. Matricial

Devemos usar embeddings pretreinados (glove) como entrada para a camada de auto-atenção. Lembrar de congelá-los pois, caso contrário,  pode ocorrer overfit.

Ao corrigir o exercicio, iremos também nos atentar na eficiencia/velocidade das implementações.

Dicas:
- A dificuldade deste exercício será implementar a auto-atenção de forma matricial usando minibatches. Para lidar com exemplos de tamanho variável, deve-se truncá-los e aplicar padding.

- Evitar usar qualquer laço na implementação matricial, pois isso a deixará muito ineficiente.

## Definindo os parametros

In [ ]:
params = {
    'vocabulary_size': 400000,
    'padding_idx': 400001,
    'max_length': 200,
}

# Fixando a seed

In [ ]:
import random
import torch
import torch.nn.functional as F
import numpy as np

In [ ]:
def set_seeds():
  random.seed(123)
  np.random.seed(123)
  torch.manual_seed(123)

set_seeds()

## Preparando Dados

Primeiro, fazemos download do dataset:

In [ ]:
!wget -nc http://files.fast.ai/data/aclImdb.tgz
!tar -xzf aclImdb.tgz

--2026-09-24 01:30:28--  http://files.fast.ai/data/aclImdb.tgz
Resolving files.fast.ai (files.fast.ai)... 172.67.69.159, 104.26.2.19, 104.26.3.19, ...
Connecting to files.fast.ai (files.fast.ai)|172.67.69.159|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://files.fast.ai/data/aclImdb.tgz [following]
--2026-09-24 01:30:28--  https://files.fast.ai/data/aclImdb.tgz
Connecting to files.fast.ai (files.fast.ai)|172.67.69.159|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 145982645 (139M) [application/x-gtar-compressed]
Saving to: ‘aclImdb.tgz’

aclImdb.tgz         100%[===================>] 139.22M   153MB/s    in 0.9s    

2026-09-24 01:30:29 (153 MB/s) - ‘aclImdb.tgz’ saved [145982645/145982645]



## Carregando o dataset

Criaremos uma divisão de treino (20k exemplos) e validação (5k exemplos) artificialmente.

In [ ]:
import os

max_valid = 5000

def load_texts(folder):
    texts = []
    for path in os.listdir(folder):
        with open(os.path.join(folder, path)) as f:
            texts.append(f.read())
    return texts

x_train_pos = load_texts('aclImdb/train/pos')
x_train_neg = load_texts('aclImdb/train/neg')
x_test_pos = load_texts('aclImdb/test/pos')
x_test_neg = load_texts('aclImdb/test/neg')

x_train = x_train_pos + x_train_neg
x_test = x_test_pos + x_test_neg
y_train = [True] * len(x_train_pos) + [False] * len(x_train_neg)
y_test = [True] * len(x_test_pos) + [False] * len(x_test_neg)

# Embaralhamos o treino para depois fazermos a divisão treino/valid.
c = list(zip(x_train, y_train))
random.shuffle(c)
x_train, y_train = zip(*c)

x_valid = x_train[-max_valid:]
y_valid = y_train[-max_valid:]
x_train = x_train[:-max_valid]
y_train = y_train[:-max_valid]

print(len(x_train), 'amostras de treino.')
print(len(x_valid), 'amostras de desenvolvimento.')
print(len(x_test), 'amostras de teste.')

print('3 primeiras amostras treino:')
for x, y in zip(x_train[:3], y_train[:3]):
    print(y, x[:100])

print('3 últimas amostras treino:')
for x, y in zip(x_train[-3:], y_train[-3:]):
    print(y, x[:100])

print('3 primeiras amostras validação:')
for x, y in zip(x_valid[:3], y_valid[:3]):
    print(y, x[:100])

print('3 últimas amostras validação:')
for x, y in zip(x_valid[-3:], y_valid[-3:]):
    print(y, x[:100])

20000 amostras de treino.
5000 amostras de desenvolvimento.
25000 amostras de teste.
3 primeiras amostras treino:
False Before the Internet this movie could never have been made but the idea that the Web is full of evil 
False Jane Russell proved to be a delightful musical-comedy performer in the similarly titled "Gentlemen P
True I was always a big fan of this movie, first of all have you seen the cast, the acting is superb and 
3 últimas amostras treino:
False I'm hearing rumors of an upcoming "Leonard Nimoy Demonstrates the Blu-ray Disc". With advances over 
True I generally find Loretta Young hard to take, too concerned with her looks and too ladylike in all th
True Okay, so the first few seasons took a while to get going on the special effects way, but from the be
3 primeiras amostras validação:
True When i first saw the title i was already deducing the theme of the film - it clearly wasn't a refere
True 'I only wanted to see you laughing in the Purple Rain.' This is an excell fil

# Carregando os embeddings do Glove

In [ ]:
!wget -nc http://nlp.stanford.edu/data/glove.6B.zip
!unzip -o glove.6B.zip -d glove_dir

--2026-09-24 01:30:42--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2026-09-24 01:30:42--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-09-24 01:30:43--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 18.8 MB/s eta 0:00:00


In [ ]:
import gensim.downloader as api

# Baixa e carrega o GloVe 6B com 300 dimensões automaticamente
glove_vectors = api.load("glove-wiki-gigaword-300")

# Uso:
try:
    print("Vetor de 'king':", glove_vectors['king'][:5])
except KeyError:
    print("Palavra não encontrada")


[==================================================] 100.0% 376.1/376.1MB downloaded
Vetor de 'king': [ 0.0033901 -0.34614    0.28144    0.48382    0.59469  ]


In [ ]:
vocab = glove_vectors.key_to_index
vocab['<UNK>'] = params['vocabulary_size'] # The last row is for the unknown token.

# We create a random vector for the unknown token
unk_vector = torch.FloatTensor(1, glove_vectors.vectors.shape[1]).uniform_(-0.5, 0.5)

# We create a vector of zeros for the pad token
pad_vector = torch.zeros(1, glove_vectors.vectors.shape[1])

glove_tensor = torch.from_numpy(glove_vectors.vectors)
embeddings = torch.cat((glove_tensor, unk_vector, pad_vector), dim=0)

vocab['<PAD>'] = params['vocabulary_size'] + 1

print(f'Total de palavras: {len(vocab)}')
print(f'embeddings.shape: {embeddings.shape}')

Total de palavras: 400002
embeddings.shape: torch.Size([400002, 300])


# Definindo o tokenizador

In [ ]:
import collections
import re


def tokenize(text):
    return [token.lower() for token in re.compile(r'\w+').findall(text)]


def to_token_ids(text, vocab, max_length, padding_idx):
    tokens = tokenize(text)[:max_length]  # Truncating.
    token_ids = []
    for token in tokens:
        # We use the id of the "<UNK>" token if we don't find it in the vocabulary.
        token_id = vocab.get(token, vocab['<UNK>'])
        token_ids.append(token_id)

    # Adding PAD tokens, if necessary.
    token_ids += [padding_idx] * max(0, max_length - len(token_ids))
    return token_ids

## Definindo o Modelo

In [ ]:
class SelfAttentionLayerLoop(torch.nn.Module):

    def __init__(self, embeddings, padding_idx):
        super(SelfAttentionLayerLoop, self).__init__()
        self.C = torch.nn.Embedding.from_pretrained(embeddings, padding_idx=padding_idx, freeze=True)
        self.padding_idx = padding_idx

    def forward(self, batch_token_ids):
        batchE = []
        for token_ids in batch_token_ids:
            valid_ids = [t for t in token_ids.tolist() if t != self.padding_idx]
            E = [self.C.weight[t] for t in valid_ids]

            tokenE = []
            for q in E:
                scores = []
                for k in E:
                    scores.append(torch.dot(q, k))
                weights = F.softmax(torch.stack(scores), dim=0)

                context = torch.zeros_like(q)
                for w, v in zip(weights, E):
                    context = context + w * v
                tokenE.append(context)

            batchE.append(torch.stack(tokenE).mean(dim=0))

        return torch.stack(batchE)

In [ ]:
class SelfAttentionLayerMatrix(torch.nn.Module):

    def __init__(self, embeddings, padding_idx):
        super(SelfAttentionLayerMatrix, self).__init__()
        self.C = torch.nn.Embedding.from_pretrained(embeddings, padding_idx=padding_idx, freeze=True)
        self.padding_idx = padding_idx

    def forward(self, batch_token_ids):
        E = self.C(batch_token_ids)

        pad_mask = (batch_token_ids == self.padding_idx)

        scores = torch.bmm(E, E.transpose(1, 2))

        scores = scores.masked_fill(pad_mask.unsqueeze(1), torch.finfo(scores.dtype).min)

        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights, E)

        non_pad = (~pad_mask).unsqueeze(-1).to(context.dtype)
        mean_embeddings = (context * non_pad).sum(dim=1) / non_pad.sum(dim=1).clamp(min=1)
        return mean_embeddings

## Testando a implementação com embeddings "falsos"

In [ ]:
fake_vocab = {
    'a': 0,
    'b': 1,
    'c': 2,
    '<UNK>': 3
}

fake_embeddings = torch.arange(0, 2 * len(fake_vocab)).reshape(len(fake_vocab), 2).float()
pad_vector = torch.zeros(1, 2)
fake_embeddings = torch.cat((fake_embeddings, pad_vector), dim=0)

fake_examples = [
    'a', # Testing PAD
    'a b',
    'a c b', # Testing truncation
    'a z', # Testing <UNK>
    ]

print(f'Total de palavras: {len(fake_vocab)}')
print(f'embeddings.shape: {fake_embeddings.shape}')

Total de palavras: 4
embeddings.shape: torch.Size([5, 2])


In [ ]:
fake_embeddings

tensor([[0., 1.],
        [2., 3.],
        [4., 5.],
        [6., 7.],
        [0., 0.]])

In [ ]:
self_attention_layer = SelfAttentionLayerLoop(
    embeddings=fake_embeddings,
    padding_idx=4)

batch_token_ids = []
for example in fake_examples:
    token_ids = to_token_ids(
        text=example,
        vocab=fake_vocab,
        max_length=2,
        padding_idx=4)
    batch_token_ids.append(token_ids)

batch_token_ids = torch.LongTensor(batch_token_ids)
my_output = self_attention_layer(batch_token_ids)

In [ ]:
target_output = torch.FloatTensor([
    [0.00000000, 1.00000000],
    [1.88075161, 2.88075161],
    [3.96402740, 4.96402740],
    [5.99258232, 6.99258232]])

In [ ]:
assert torch.allclose(my_output, target_output, atol=1e-6)

In [ ]:
self_attention_layer = SelfAttentionLayerMatrix(
    embeddings=fake_embeddings,
    padding_idx=4)

batch_token_ids = []
for example in fake_examples:
    token_ids = to_token_ids(
        text=example,
        vocab=fake_vocab,
        max_length=2,
        padding_idx=4)
    batch_token_ids.append(token_ids)

batch_token_ids = torch.LongTensor(batch_token_ids)
my_output = self_attention_layer(batch_token_ids)

In [ ]:
assert torch.allclose(my_output, target_output, atol=1e-6)

## Testando a implementação com 8 exemplos do dataset do IMDB

In [ ]:
examples = [
    "THE TEMP (1993) didn't do much theatrical business, but here's the direct-to-video rip-off you didn't want, anyway! Ellen Bradford (Mel Harris) is the new woman at Millennium Investments, a high scale brokerage firm, who starts getting helpful hints from wide-eyed secretary Deidre (Sheila Kelley). Deidre turns out to be an ambitious daddy's girl who will stop at nothing to move up the corporate ladder, including screwing a top broker she can't stand and murdering anyone who gets on her bad side. She digs up skeletons in Ellen's closet, tries to cause problems with her husband (Barry Bostwick), kills while making it look like she is responsible, kidnaps her daughter and tries to get her to embezzle money from the company.<br /><br />Harris and Kelley deliver competent performances, the supporting cast is alright and it's reasonably well put-together, but that doesn't fully compensate for a script that travels down a well-worn path and offers few surprises.",
    "Sondra Locke stinks in this film, but then she was an awful 'actress' anyway. Unfortunately, she drags everyone else (including then =real life boyfriend Clint Eastwood down the drain with her. But what was Clint Eastwood thinking when he agreed to star in this one? One read of the script should have told him that this one was going to be a real snorer. It's an exceptionally weak story, basically no story or plot at all. Add in bored, poor acting, even from the normally good Eastwood. There's absolutely no action except a couple arguments and as far as I was concerned, this film ranks up at the top of the heap of natural sleep enhancers. Wow! Could a film BE any more boring? I think watching paint dry or the grass grow might be more fun. A real stinker. Don't bother with this one.",
    "Judy Davis shows us here why she is one of Australia's most respected and loved actors - her portrayal of a lonely, directionless nomad is first-rate. A teenaged Claudia Karvan also gives us a glimpse of what would make her one of this country's most popular actors in years to come, with future roles in THE BIG STEAL, THE HEARTBREAK KID, DATING THE ENEMY, RISK and the acclaimed TV series THE SECRET LIFE OF US. (Incidentally, Karvan, as a child, was a young girl whose toy Panda was stolen outside a chemist's shop in the 1983 drama GOING DOWN with Tracey Mann.) If this films comes your way, make sure you see it!! Rating: 79/100. See also: HOTEL SORRENTO, RADIANCE, VACANT POSSESSION, LANTANA.",
    'New York playwright Michael Caine (as Sidney Bruhl) is 46-years-old and fading fast; as the film opens, Mr. Caine\'s latest play flops on Broadway. TV reviewers poke fun at Caine, and he gets drunk. Passing out on the Long Island Railroad lands Caine in Montauk, instead of his residence in East Hampton. Finally arriving home, Caine is comforted by tightly-attired wife Dyan Cannon (as Myra), an unfortunately high-strung heart patient. There, Caine and Ms. Cannon discuss a new play called "Deathtrap", written by hunky young Christopher Reeve (as Clifford "Cliff" Anderson), one of Caine\'s former students. The couple believe Mr. Reeve\'s "Deathtrap" is the hit needed to revive Caine\'s career.<br /><br />"The Trap Is Set\x85 For A Wickedly Funny Who\'ll-Do-It." <br /><br />Directed by Sidney Lumet, Ira Levin\'s long-running Broadway hit doesn\'t stray too far from its stage origin. The cast is enjoyable and the story\'s twists are still engrossing. One thing that did not work (for me) was the curtain call ending; surely, it played better on stage. "Deathtrap" is a fun film to watch again; the performances are dead on - but, in hindsight, the greeting Reeve gives Caine at the East Hampton train station should have been simplified to a smiling "Hello." The location isn\'t really East Hampton, but the windmill and pond look similar. And, the much ballyhooed love scene is shockingly tepid. But, the play was so good, "even a gifted director couldn\'t ruin it." And, Mr. Lumet doesn\'t disappoint.<br /><br />******** Deathtrap (3/19/82) Sidney Lumet ~ Michael Caine, Christopher Reeve, Dyan Cannon, Irene Worth',
    'Students often ask me why I choose this version of Othello. Shakespeare\'s text is strongly truncated and the film contains material which earned it an "R" rating.<br /><br />I have several reasons for using this production: First, I had not seen a depiction of the Moor that actually made me sympathetic to Othello until I saw Fishburne play him. I saw James Earl Jones and Christopher Plummer play Othello and Iago on Broadway, and it was wonderful. Plummer\'s energy was especially noticeable. But in spite of Jone\'s incredible presence both physically and vocally, the character he played just seemed too passive to illicit from me a complete emotional purgation in the Aristotelian sense. Jones, in fact, affirmed what I felt when in an interview he noted that he had played Othello as passive--seeing Iago as basically doing him over. Unfortunately this sapped my grief for the character destruction. Thus, I felt sympathy for Jone\'s Moor but not the horror over his corruption by an evil man. In contrast, Fishburne\'s Othello is a strong and vigorous figure familiar with taking action. Thus, Iago\'s temptation to actively deal with what is presented to Othello as his wife\'s unfaithfulness is a perversion of the general\'s positive quality to be active not passive.1 The horror of the story is that this good quality in Othello becomes perverted. Fishburne\'s depiction is therefore classically tragic.<br /><br />Second, Fishburne is the first black actor to play Othello in a film. Both Orsen Wells and Anthony Hopkins did fine film versions, but they were white men in black face.2 Why is this important? Why should a Black actor be the Black man on the stage?3 Certainly in Shakespeare\'s day they used black face just as they used boys to make girls. Perhaps then, the reason is the same. Female actors bring a special quality to female roles on the Shakespearian stage because they understand best what Shakespeare\'s genius was trying to present. A gifted black actor should play the moor because his experience in a white dominated culture is vital to understanding what Shakespeare\'s genius recognized: the pain of being marginalized because of race. An important theme in Othello is isolation caused by racism. Although it is a mistake to insert American racism into a Shakespearian play, there can be little doubt that racism is still working among the characters. Many, including Desdimona\'s father, think that a union between a Venetian white Christian woman and a North African black Christian man is UNNATURAL.<br /><br />Third, Shakespeare was never G rated. He never has been. His stage productions were always typified by violence and strong language. But Shakespeare\'s genius uses these elements not as sensationialism but for artistic honesty.',
    'Roeg has done some great movies, but this a turkey. It has a feel of a play written by an untalented high-school student for his class assignment. The set decoration is appealing in a somewhat surrealistic way, but the actual story is insufferable hokum.',
    "<br /><br />What is left of Planet Earth is populated by a few poor and starving rag-tag survivors. They must eat bugs and insects, or whatever, after a poison war, or something, has nearly wiped out all human civilization. In these dark times, one of the few people on Earth still able to live in comfort, we will call him the All Knowing Big Boss, has a great quest to prevent some secret spore seeds from being released into the air. It seems that the All Knowing Big Boss is the last person on Earth that knows that these spores even exist. The spores are located far away from any living soul, and they are highly protected by many layers of deadly defense systems. <br /><br />The All Knowing Big Boss wants the secret spores to remain in their secret protected containers. So, he makes a plan to send in a macho action team to remove the spore containers from all of the protective systems and secret location. Sending people to the location of secret spores makes them no longer a secret. Sending people to disable all of the protective systems makes it possible for the spores to be easily released into the air. How about letting sleeping dogs lie?! <br /><br />The one pleasant feature of ENCRYPT is the radiant and elegant Vivian Wu. As the unremarkable macho action team members drop off with mechanically paced predictable timing, engaging Vivian Wu's charm makes acceptable the plot idea of her old employer wanting her so much. She is an object of love, an object of desire -- a very believable concept!<br /><br />Fans of Vivian Wu may want to check out an outstanding B-movie she is in from a couple years back called DINNER RUSH. DINNER RUSH is highly recommended. ENCRYPT is not.",
    "So the other night I decided to watch Tales from the Hollywood Hills: Natica Jackson. Or Power, Passion, Murder as it is called in Holland. When I bought the film I noticed that Michelle Pfeiffer was starring in it and I thought that had to say something about the quality. Unfortunately, it didn't.<br /><br />1) The plot of the film is really confusing. There are two story lines running simultaneously during the film. Only they have nothing in common. Throughout the entire movie I was waiting for the moment these two story lines would come together so the plot would be clear to me. But it still hasn't.<br /><br />2) The title of the film says the film will be about Natica Jackson. Well it is, sometimes. Like said the film covers two different stories and the part about Natica Jackson is the shortest. So another title for this movie would not be a wrong choice.<br /><br />To conclude my story, I really recommend that you leave this movie where it belongs, on the shelf in the store on a place nobody can see it. By doing this you won't waste 90 minutes of your life, as I did."
]

In [ ]:
%%time
self_attention_layer = SelfAttentionLayerLoop(
    embeddings=embeddings,
    padding_idx=params['padding_idx'])

batch_token_ids = []
for example in examples:
    token_ids = to_token_ids(
        text=example,
        vocab=vocab,
        max_length=params['max_length'],
        padding_idx=params['padding_idx'])
    batch_token_ids.append(token_ids)

batch_token_ids = torch.LongTensor(batch_token_ids)
my_output = self_attention_layer(batch_token_ids)
loop_output = my_output

CPU times: user 3.68 s, sys: 14.6 ms, total: 3.69 s
Wall time: 6.93 s


In [ ]:
%%time
self_attention_layer = SelfAttentionLayerMatrix(
    embeddings=embeddings,
    padding_idx=params['padding_idx'])

batch_token_ids = []
for example in examples:
    token_ids = to_token_ids(
        text=example,
        vocab=vocab,
        max_length=params['max_length'],
        padding_idx=params['padding_idx'])
    batch_token_ids.append(token_ids)

batch_token_ids = torch.LongTensor(batch_token_ids)
my_output = self_attention_layer(batch_token_ids)

CPU times: user 10.2 ms, sys: 5.95 ms, total: 16.2 ms
Wall time: 67.1 ms


In [ ]:
assert torch.allclose(my_output, loop_output, atol=1e-4)

# Env config

In [ ]:
if torch.cuda.is_available():
   dev = "cuda:0"
   print(torch. cuda. get_device_name(dev))
else:
   dev = "cpu"
print(dev)
device = torch.device(dev)

Tesla T4
cuda:0


# Modelo completo

Configure agora o dataset e o modelo completo para treinamento.

# Dataset

In [ ]:
class Ex4_ds(torch.utils.data.Dataset):
    def __init__(self, x, y, vocabulary, max_length, padding_idx):
        self.x = torch.LongTensor([
            to_token_ids(x[i], vocabulary, max_length, padding_idx) for i in range(len(x))])
        self.y = torch.LongTensor([int(y[i]) for i in range(len(y))])

    def __len__(self):
        return len(self.x)

    def __getitem__(self, index):
        return self.x[index], self.y[index]

# Model

In [ ]:
class Ex4_model(torch.nn.Module):
    def __init__(self, input, hidden, output, embeddings, padding_idx):
        super(Ex4_model, self).__init__()
        self.attention = SelfAttentionLayerMatrix(embeddings, padding_idx)
        self.hidden_layer = torch.nn.Linear(input, hidden)
        self.activation = torch.nn.ReLU()
        self.output_layer = torch.nn.Linear(hidden, output)

    def forward(self, x):
        x = self.attention(x)
        x = self.hidden_layer(x)
        x = self.activation(x)
        x = self.output_layer(x)
        return x


In [ ]:
hyperparameters = { "mode": "Aula4",
          "learning_rate": 1e-2,
          "n_epochs": 5,
          "batch_size": 50,
          "hidden_size": 128,
          "input_size": glove_vectors.vectors.shape[1],
          "output_size": 2 }
model = Ex4_model(hyperparameters["input_size"], hyperparameters["hidden_size"],
                  hyperparameters["output_size"], embeddings, params['padding_idx'])
model.to(device)
print(model)

Ex4_model(
  (attention): SelfAttentionLayerMatrix(
    (C): Embedding(400002, 300, padding_idx=400001)
  )
  (hidden_layer): Linear(in_features=300, out_features=128, bias=True)
  (activation): ReLU()
  (output_layer): Linear(in_features=128, out_features=2, bias=True)
)


# Train Loop

In [ ]:
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
def train_loop(dataloader_train, dataloader_val, hyperparameters, model):
    min_val_loss = 10e9
    best_model = 'best_model.pt'

    # Gradient descent
    optimizer = torch.optim.Adam(model.parameters(), lr=hyperparameters['learning_rate'])
    best_epoch = 0

    for epoch in range(hyperparameters['n_epochs']):
      train_loss = 0
      model.train()
      for x_train, y_train in dataloader_train:
            # transform to one dimmension
        x_train = x_train.to(device)
        y_train = y_train.to(device)

        outputs = model(x_train)

            # batch loss
        batch_loss = criterion(outputs, y_train)

            # reset gradients, backpropagation, optimizer step and sum loss
        optimizer.zero_grad()
        batch_loss.backward()
        optimizer.step()
        train_loss += batch_loss.item()
            #print(f'{hyperparameters["name"]}_train/batch_loss: {batch_loss}')

      train_loss = train_loss / len(dataloader_train.dataset)
        #print(f'Epoch {epoch} / {hyperparameters["name"]} train loss: {train_loss}')

        # Validation (end of epoch).
      total_loss = 0
      total_acc = 0
      model.eval()
      with torch.no_grad():
        for x_val, y_val in dataloader_val:
          x_val = x_val.to(device)
          y_val = y_val.to(device)

                # predict
          outputs = model(x_val)

                # batch loss
          batch_loss = criterion(outputs, y_val)
          preds = outputs.argmax(dim=1)
          total_loss += batch_loss.item()

          batch_acc = (preds == y_val).sum()
          total_acc += batch_acc
      val_loss = total_loss / len(dataloader_val.dataset)

      val_acc = total_acc / len(dataloader_val.dataset)

      print(f'Model: {hyperparameters["mode"]}, Epoch: {epoch+1}/{hyperparameters["n_epochs"]} - train_loss: {train_loss} - val_loss: {val_loss} - acc: {total_acc / len(dataloader_val.dataset)*100} %')

        # Save best model
      if val_loss < min_val_loss:
        torch.save(model.state_dict(), 'best_model.pt')
        min_val_loss = val_loss
        best_epoch = epoch
        print(f'Model: {hyperparameters["mode"]} - best model in epoch: {best_epoch+1}')


In [ ]:
def predict(model, dataloader_test):
    best_model = 'best_model.pt'
    model.load_state_dict(torch.load(best_model))
    model.eval()
    model.to(device)
    floss = 0
    total_acc = 0
    with torch.no_grad():
      for x_t, y_t in dataloader_test:
        x_t = x_t.to(device)
        y_t = y_t.to(device)

        outputs = model(x_t)
        loss = criterion(outputs, y_t)
        floss += loss
        pred = outputs.argmax(dim=1)
        batch_acc = (pred == y_t).sum()
        total_acc += batch_acc

      test_acc = total_acc / len(dataloader_test.dataset)
    return {
        'loss':  floss / len(dataloader_test.dataset),
        'acc': test_acc
    }

# List to dict

In [ ]:
# Transform list to dict
x_train = {num: i for num, i in enumerate(x_train)}
y_train = {num: i for num, i in enumerate(y_train)}
x_valid = {num: i for num, i in enumerate(x_valid)}
y_valid = {num: i for num, i in enumerate(y_valid)}
x_test = {num: i for num, i in enumerate(x_test)}
y_test = {num: i for num, i in enumerate(y_test)}

# Training

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
train_ds = Ex4_ds(x_train, y_train, vocab, params['max_length'], params['padding_idx'])
val_ds = Ex4_ds(x_valid, y_valid, vocab, params['max_length'], params['padding_idx'])
dataloader_train = DataLoader(train_ds, batch_size=hyperparameters['batch_size'], shuffle=True)
dataloader_val = DataLoader(val_ds, batch_size=hyperparameters['batch_size'], shuffle=False)

In [ ]:
train_loop(dataloader_train, dataloader_val, hyperparameters, model)

Model: Aula4, Epoch: 1/5 - train_loss: 0.009509947386384011 - val_loss: 0.008673520469665527 - acc: 79.5199966430664 %
Model: Aula4 - best model in epoch: 1
Model: Aula4, Epoch: 2/5 - train_loss: 0.00862269890010357 - val_loss: 0.008276884981989861 - acc: 81.18000030517578 %
Model: Aula4 - best model in epoch: 2
Model: Aula4, Epoch: 3/5 - train_loss: 0.008595287666469812 - val_loss: 0.00870807797908783 - acc: 79.91999816894531 %
Model: Aula4, Epoch: 4/5 - train_loss: 0.008389613729715348 - val_loss: 0.008588906383514405 - acc: 80.93999481201172 %
Model: Aula4, Epoch: 5/5 - train_loss: 0.008336745814979077 - val_loss: 0.00841351347565651 - acc: 80.7199935913086 %


In [ ]:
del train_ds
del val_ds
del dataloader_train
del dataloader_val

In [ ]:
test_ds = Ex4_ds(x_test, y_test, vocab, params['max_length'], params['padding_idx'])
dataloader_test = DataLoader(test_ds, batch_size=hyperparameters['batch_size'], shuffle=False)
print(predict(model,dataloader_test))

{'loss': tensor(0.0084, device='cuda:0'), 'acc': tensor(0.8067, device='cuda:0')}


Utilização do Claude para ajuda no erros com o código
